#  OBJECTIVE

**This notebook focuses on building the AirFair-Vista Streamlit web application** — connecting the serialised ML pipeline to a user-facing interface through a cleanly separated backend (`preprocessor.py`) and frontend (`app.py`) architecture, with Cloudflare tunnelling for public access from Google Colab.

> **Architecture:** `preprocessor.py` (feature engineering + prediction logic) ← imported by → `app.py` (Streamlit UI with Plotly charts) | **Served via:** Cloudflare tunnel on port 8501

---
##  Step: Environment Setup — Model Verification & Package Installation

**Why:** Before writing any application code, verifying the model file exists and has a non-zero size catches deployment errors early (corrupted serialisation, wrong path). Installing `streamlit`, `scikit-learn`, and `joblib` in the Colab environment ensures the app runtime matches the training environment — preventing the silent version mismatch failures that are the #1 cause of ML deployment bugs.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/AirFair-Vista/app', exist_ok=True)

# Confirm model exists
MODEL_PATH = '/content/drive/MyDrive/AirFair-Vista/models/flight_price_prediction_pipeline.pkl'    #content model +feature
if os.path.exists(MODEL_PATH):
    size_mb = os.path.getsize(MODEL_PATH) / (1024*1024)
    print(f' Model found: {MODEL_PATH} ({size_mb:.1f} MB)')
else:
    print(f' Model not found at: {MODEL_PATH}')
    print('   Upload it to Drive first.')

Mounted at /content/drive
 Model found: /content/drive/MyDrive/AirFair-Vista/models/flight_price_prediction_pipeline.pkl (53.2 MB)


In [ ]:
import joblib

pkl = joblib.load("/content/drive/MyDrive/AirFair-Vista/models/flight_price_prediction_pipeline.pkl")

print(type(pkl))
print(pkl)

<class 'dict'>
{'model': RandomForestRegressor(max_depth=12, n_estimators=300, random_state=42), 'features': ['journey_day', 'journey_month', 'journey_weekday', 'is_weekend', 'quarter', 'dep_hour', 'weekday', 'is_holiday', 'duration_hours', 'duration_minutes', 'total_duration_mins', 'Source_freq', 'Destination_freq', 'Airline_mean_price', 'Source_mean_price', 'total_duration_mins.1', 'journey_month.1', 'total_duration_mins^2', 'total_duration_mins journey_month', 'journey_month^2'], 'model_name': 'RandomForest_FlightPrice', 'version': '1.0'}


In [ ]:
!pip install streamlit scikit-learn joblib -q
print(' Packages installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 108.3 MB/s eta 0:00:00
 Packages installed


---
##  Step: Backend — `preprocessor.py` (Prediction Engine)

**Why:** All feature engineering logic (airline frequency lookup, haversine distance calculation, holiday flagging, duration prediction, feature matrix construction) is isolated in this module for three reasons: (1) Streamlit reruns `app.py` on every widget change — putting heavy logic there wastes compute; (2) `preprocessor.py` can be imported and unit-tested without launching the Streamlit server; (3) future REST API or batch scoring consumers can reuse the identical preprocessing pipeline, guaranteeing consistency between training and serving features.

**Key design decisions:**
- `@st.cache_resource` for model loading: creates a shared singleton across all users (correct for a 53MB model object), unlike `cache_data` which creates per-argument copies.
- `build_feature_matrix()` constructs the exact same feature vector the Random Forest was trained on — column order, encoding, and derived features are all reproduced identically.
- `haversine_km()` computes great-circle distance between city coordinates as a proxy for route length — an unseen feature not in the training data that enriches predictions for routes not well-represented in the training set.

---
##  Step: Frontend — `app.py` (Streamlit User Interface)

**Why:** The Streamlit app is the product — it translates the ML model into a tool real users can interact with. The UI design follows the prediction workflow: (1) user selects flight attributes via dropdowns/sliders → (2) backend validates inputs → (3) `predict_price()` is called → (4) result displayed with Plotly price gauge and comparison charts. Plotly is used over matplotlib because Streamlit renders it natively as interactive HTML — users can hover, zoom, and explore the price visualisations without any page reload.

In [ ]:
%%writefile /content/drive/MyDrive/AirFair-Vista/backend/preprocessor.py
"""
AirFair-Vista — Preprocessing Pipeline
pipeline/preprocessor.py

WHY THIS FILE EXISTS:
  All feature engineering, lookup tables, distance calculation, validation
  rules and prediction functions live here — completely separate from the
  Streamlit UI (app.py).

WHY THIS SEPARATION:
  1. app.py runs on every widget change; heavy logic should NOT be there.
  2. This file can be unit-tested independently (no Streamlit required).
  3. Other consumers (batch scoring, REST API) can import the same pipeline.
  4. Streamlit caches the imported module — zero re-execution on reruns.

USAGE IN app.py:
  import sys
  sys.path.insert(0, '/content/drive/MyDrive/AirFair-Vista/pipeline')
  from preprocessor import (
      AIRLINES, SOURCES, DESTINATIONS, STOPS, MONTHS, WEEKDAYS,
      VALID_ROUTES, VALID_DESTINATIONS, VALID_AIRLINE_STOPS,
      MAX_PAX_BY_STOPS, INDIAN_HOLIDAYS, CITY_COORDS,
      PRICE_MIN, PRICE_MAX, PRICE_AVG, PRICE_MED,
      MODEL_FEATURES, MODEL_PATH,
      haversine_km, is_indian_holiday, predict_duration,
      get_validation_errors, build_features, build_feature_matrix,
      batch_predict
  )
"""

import math
import numpy as np
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
#  SECTION 1 — DROPDOWN OPTIONS
#  Source: real values from flight_price_feature_engineered.csv
# ─────────────────────────────────────────────────────────────────────────────
AIRLINES = [
    'Air Asia', 'Air India', 'GoAir', 'IndiGo', 'Jet Airways',
    'Jet Airways Business', 'Multiple Carriers',
    'Multiple Carriers Premium Economy',
    'SpiceJet', 'TruJet', 'Vistara', 'Vistara Premium Economy'
]
SOURCES      = ['Banglore', 'Chennai', 'Delhi', 'Kolkata', 'Mumbai']
DESTINATIONS = ['Banglore', 'Cochin', 'Delhi', 'Hyderabad', 'Kolkata', 'New Delhi']
STOPS        = ['non-stop', '1 stop', '2 stops', '3 stops', '4 stops']
MONTHS       = {3: 'March', 4: 'April', 5: 'May', 6: 'June'}
WEEKDAYS     = ['Monday', 'Tuesday', 'Wednesday', 'Thursday',
                'Friday', 'Saturday', 'Sunday']

# ─────────────────────────────────────────────────────────────────────────────
#  SECTION 2 — FEATURE LOOKUP TABLES
#  Source: dataset groupby means — used to build numeric model input.
#  WHY pre-computed means instead of one-hot encoding:
#    The original training used target-encoded / frequency-encoded features,
#    not dummy variables. We replicate that exact encoding here.
# ─────────────────────────────────────────────────────────────────────────────
AIRLINE_MEAN_PRICE = {
    'Air Asia': 5590.26,        'Air India': 9612.43,
    'GoAir': 5861.06,           'IndiGo': 5673.68,
    'Jet Airways': 11643.92,    'Jet Airways Business': 58358.67,
    'Multiple Carriers': 10902.68,
    'Multiple Carriers Premium Economy': 11418.85,
    'SpiceJet': 4338.28,        'TruJet': 4140.00,
    'Vistara': 7796.35,         'Vistara Premium Economy': 8962.33
}
SOURCE_FREQ = {
    'Banglore': 0.2057, 'Chennai': 0.0357, 'Delhi': 0.4246,
    'Kolkata': 0.2688,  'Mumbai': 0.0652
}
DEST_FREQ = {
    'Banglore': 0.2688, 'Cochin': 0.4246,  'Delhi': 0.1184,
    'Hyderabad': 0.0652,'Kolkata': 0.0357,  'New Delhi': 0.0872
}
SOURCE_MEAN_PRICE = {
    'Banglore': 8017.46, 'Chennai': 4789.89, 'Delhi': 10540.11,
    'Kolkata': 9158.39,  'Mumbai': 5059.71
}

# Overall dataset price statistics
PRICE_MIN = 1759
PRICE_MAX = 79512
PRICE_AVG = 9087
PRICE_MED = 8372

# Fallback blended means (used if model.pkl not found on Drive)
AIRLINE_MEAN = {
    'Air Asia': 5590,  'Air India': 9612,   'GoAir': 5861,
    'IndiGo': 5674,    'Jet Airways': 11644, 'Jet Airways Business': 58359,
    'Multiple Carriers': 10903, 'Multiple Carriers Premium Economy': 11419,
    'SpiceJet': 4338,  'TruJet': 4140,      'Vistara': 7796,
    'Vistara Premium Economy': 8962
}
STOPS_MEAN = {
    'non-stop': 5025, '1 stop': 10594,
    '2 stops': 12716, '3 stops': 13112, '4 stops': 17686
}
ROUTE_MEAN = {
    ('Banglore','Delhi'): 5144,   ('Banglore','New Delhi'): 11918,
    ('Chennai','Kolkata'): 4790,  ('Delhi','Cochin'): 10540,
    ('Kolkata','Banglore'): 9158, ('Mumbai','Hyderabad'): 5060
}
HOUR_MEAN = {
    0:7615, 1:4355, 2:8420, 3:10475, 4:7252, 5:9682,
    6:8314, 7:8496, 8:10083, 9:9644, 10:8928, 11:9290,
    12:9252,13:9064,14:9906, 15:7687, 16:10320,17:8737,
    18:10036,19:8485,20:9671,21:8456, 22:7858, 23:9474
}
MONTH_MEAN   = {3:10673, 4:5771, 5:9128, 6:8829}
WEEKDAY_MEAN = {0:8500,1:9026,2:9278,3:8931,4:9718,5:8973,6:9526}

# ─────────────────────────────────────────────────────────────────────────────
#  SECTION 3 — DURATION PREDICTION
#  WHY lookup-first approach over pure Haversine:
#    Real flight durations include taxi, ATC delays, actual routing.
#    A lookup of real dataset means is more accurate than distance/speed.
#    Haversine is only the fallback for unknown route+stops combos.
# ─────────────────────────────────────────────────────────────────────────────
DURATION_LOOKUP = {
    ('Banglore',  'Delhi',     'non-stop'): 2.22,
    ('Banglore',  'New Delhi', 'non-stop'): 2.03,
    ('Banglore',  'New Delhi', '1 stop'):   12.88,
    ('Banglore',  'New Delhi', '2 stops'):  21.87,
    ('Banglore',  'New Delhi', '3 stops'):  22.86,
    ('Banglore',  'New Delhi', '4 stops'):  29.00,
    ('Chennai',   'Kolkata',   'non-stop'): 2.00,
    ('Delhi',     'Cochin',    'non-stop'): 2.85,
    ('Delhi',     'Cochin',    '1 stop'):   11.35,
    ('Delhi',     'Cochin',    '2 stops'):  20.19,
    ('Delhi',     'Cochin',    '3 stops'):  27.28,
    ('Kolkata',   'Banglore',  'non-stop'): 2.00,
    ('Kolkata',   'Banglore',  '1 stop'):   14.60,
    ('Kolkata',   'Banglore',  '2 stops'):  19.53,
    ('Kolkata',   'Banglore',  '3 stops'):  23.18,
    ('Mumbai',    'Hyderabad', 'non-stop'): 1.00,
    ('Mumbai',    'Hyderabad', '1 stop'):   16.02,
    ('Mumbai',    'Hyderabad', '2 stops'):  17.93,
    ('Mumbai',    'Hyderabad', '3 stops'):  24.00,
}

# Real avg speeds from dataset (dist_km / duration_hours per stop type)
AVG_SPEED_BY_STOPS = {
    'non-stop': 756.8, '1 stop': 198.9,
    '2 stops':  116.8, '3 stops': 93.8, '4 stops': 60.0
}

# Real GPS airport coordinates (used by Haversine fallback)
CITY_COORDS = {
    'Banglore':  (12.9716,  77.5946),
    'Chennai':   (13.0827,  80.2707),
    'Delhi':     (28.7041,  77.1025),
    'New Delhi': (28.6139,  77.2090),
    'Kolkata':   (22.5726,  88.3639),
    'Mumbai':    (19.0760,  72.8777),
    'Cochin':    ( 9.9312,  76.2673),
    'Hyderabad': (17.3850,  78.4867),
}

# ─────────────────────────────────────────────────────────────────────────────
#  SECTION 4 — VALIDATION CONSTANTS
#  All derived from real dataset — not invented.
# ─────────────────────────────────────────────────────────────────────────────
VALID_ROUTES = {
    ('Banglore', 'Delhi'),    ('Banglore', 'New Delhi'),
    ('Chennai',  'Kolkata'),  ('Delhi',    'Cochin'),
    ('Kolkata',  'Banglore'), ('Mumbai',   'Hyderabad'),
}
VALID_DESTINATIONS = {
    'Banglore': ['Delhi', 'New Delhi'],
    'Chennai':  ['Kolkata'],
    'Delhi':    ['Cochin'],
    'Kolkata':  ['Banglore'],
    'Mumbai':   ['Hyderabad'],
}
VALID_AIRLINE_STOPS = {
    'Air Asia':                           ['non-stop', '1 stop', '2 stops'],
    'Air India':                          ['non-stop', '1 stop', '2 stops', '3 stops', '4 stops'],
    'GoAir':                              ['non-stop', '1 stop'],
    'IndiGo':                             ['non-stop', '1 stop', '2 stops'],
    'Jet Airways':                        ['non-stop', '1 stop', '2 stops'],
    'Jet Airways Business':               ['1 stop', '2 stops'],
    'Multiple Carriers':                  ['1 stop', '2 stops', '3 stops'],
    'Multiple Carriers Premium Economy':  ['1 stop'],
    'SpiceJet':                           ['non-stop', '1 stop'],
    'TruJet':                             ['1 stop'],
    'Vistara':                            ['non-stop', '1 stop'],
    'Vistara Premium Economy':            ['non-stop'],
}
MAX_PAX_BY_STOPS = {
    'non-stop': 9, '1 stop': 9, '2 stops': 6, '3 stops': 4, '4 stops': 2
}

# Indian public holidays (fixed-date, Mar–Jun 2019 — dataset period)
INDIAN_HOLIDAYS = {
    (3, 25): 'Holi',
    (3, 29): 'Good Friday',
    (4, 14): 'Ambedkar Jayanti / Tamil New Year',
    (4, 17): 'Ram Navami',
    (4, 19): 'Mahavir Jayanti',
    (5,  1): 'Labour Day',
    (5, 23): 'Buddha Purnima',
    (6,  5): 'Eid ul-Fitr',
}

# ─────────────────────────────────────────────────────────────────────────────
#  SECTION 5 — MODEL CONFIG
#  Exact 20-feature order from pkl['features'] — must never change order.
# ─────────────────────────────────────────────────────────────────────────────
MODEL_PATH = '/content/drive/MyDrive/AirFair-Vista/models/flight_price_prediction_pipeline.pkl'

MODEL_FEATURES = [
    'journey_day', 'journey_month', 'journey_weekday', 'is_weekend', 'quarter',
    'dep_hour', 'weekday', 'is_holiday', 'duration_hours', 'duration_minutes',
    'total_duration_mins', 'Source_freq', 'Destination_freq',
    'Airline_mean_price', 'Source_mean_price',
    'total_duration_mins.1', 'journey_month.1',
    'total_duration_mins^2', 'total_duration_mins journey_month', 'journey_month^2'
]

# ─────────────────────────────────────────────────────────────────────────────
#  SECTION 6 — PURE FUNCTIONS (no Streamlit dependency)
#  WHY no st.* calls here: this file must be importable without Streamlit.
#  Tests, batch scripts and API servers should all work with this module.
# ─────────────────────────────────────────────────────────────────────────────

def haversine_km(city1: str, city2: str) -> float:
    """
    Great-circle distance in km between two cities using Haversine formula.
    WHY Haversine not Euclidean: Earth is not flat — Euclidean is wrong at scale.
    WHY not an API (Google Maps): no network call needed; we have fixed airports.
    """
    if city1 == city2:
        return 0.0
    c1 = CITY_COORDS.get(city1)
    c2 = CITY_COORDS.get(city2)
    if not c1 or not c2:
        return 0.0
    R    = 6371.0
    lat1, lon1 = math.radians(c1[0]), math.radians(c1[1])
    lat2, lon2 = math.radians(c2[0]), math.radians(c2[1])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a    = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return round(2 * R * math.asin(math.sqrt(a)), 1)


def is_indian_holiday(month: int, day: int) -> int:
    """Returns 1 if date is a recognised Indian public holiday, else 0."""
    return 1 if (month, day) in INDIAN_HOLIDAYS else 0


def predict_duration(source: str, destination: str, stops: str) -> float:
    """
    Predict flight duration in hours.
    Priority:
      1. Exact dataset lookup (most accurate — real observed means)
      2. Haversine ÷ avg speed for stop type (realistic physics-based)
      3. Global dataset mean 10.2h (last resort)
    WHY not a fixed formula: stops add layover time, not just distance.
    """
    key = (source, destination, stops)
    if key in DURATION_LOOKUP:
        return DURATION_LOOKUP[key]
    dist  = haversine_km(source, destination)
    speed = AVG_SPEED_BY_STOPS.get(stops, 300.0)
    if dist > 0 and speed > 0:
        return max(0.5, round(dist / speed, 1))
    return 10.2


def get_validation_errors(source, destination, airline, stops, passengers, dep_hour):
    """
    Returns (errors, warnings) lists.
    errors  → block submission (invalid inputs)
    warnings → allow submission but flag low confidence
    All 8 rules are derived from real dataset analysis — nothing invented.
    """
    errors, warnings = [], []

    if source.lower() == destination.lower():
        errors.append('Source and destination cannot be the same city.')
    elif (source, destination) not in VALID_ROUTES:
        rev  = (destination, source)
        hint = f' Try {destination} → {source} instead.' if rev in VALID_ROUTES else ''
        errors.append(f'Route {source} → {destination} has no records in the dataset.{hint}')

    valid_stops_for_airline = VALID_AIRLINE_STOPS.get(airline, [])
    if stops not in valid_stops_for_airline:
        errors.append(
            f'{airline} does not operate {stops} flights in the dataset. '
            f'Valid options: {", ".join(valid_stops_for_airline)}.'
        )

    max_pax = MAX_PAX_BY_STOPS.get(stops, 9)
    if passengers > max_pax:
        errors.append(
            f'{stops} flights support max {max_pax} passengers. '
            f'Reduce to {max_pax} or choose fewer stops.'
        )

    if airline == 'Jet Airways Business' and stops == 'non-stop':
        warnings.append('Jet Airways Business has no non-stop records. Prediction may be less accurate.')
    if dep_hour in [2, 3, 4]:
        warnings.append(f'Departure at {dep_hour:02d}:00 is a red-eye slot — low data confidence.')
    if airline == 'TruJet':
        warnings.append('TruJet has only 1 flight record in the dataset.')
    if airline == 'Multiple Carriers Premium Economy':
        warnings.append('Multiple Carriers Premium Economy has only 13 records.')

    return errors, warnings


def build_features(airline, source, destination,
                   dep_hour, journey_month, journey_weekday,
                   journey_day, duration_hours) -> pd.DataFrame:
    """
    Builds the exact 20-column numeric DataFrame for ONE prediction.
    WHY DataFrame (not numpy array): model was fitted with named columns;
    sklearn RF uses feature names for validation — named input avoids warnings.
    Column order enforced by MODEL_FEATURES to guarantee correct alignment.
    """
    tm  = duration_hours * 60.0
    dm  = (duration_hours % 1) * 60.0
    row = {
        'journey_day':                      journey_day,
        'journey_month':                    journey_month,
        'journey_weekday':                  journey_weekday,
        'is_weekend':                       1 if journey_weekday >= 5 else 0,
        'quarter':                          (journey_month - 1) // 3 + 1,
        'dep_hour':                         dep_hour,
        'weekday':                          journey_weekday,
        'is_holiday':                       0,      # always 0 in training data
        'duration_hours':                   duration_hours,
        'duration_minutes':                 dm,
        'total_duration_mins':              tm,
        'Source_freq':                      SOURCE_FREQ.get(source, 0.2),
        'Destination_freq':                 DEST_FREQ.get(destination, 0.2),
        'Airline_mean_price':               AIRLINE_MEAN_PRICE.get(airline, PRICE_AVG),
        'Source_mean_price':                SOURCE_MEAN_PRICE.get(source, PRICE_AVG),
        'total_duration_mins.1':            tm,
        'journey_month.1':                  journey_month,
        'total_duration_mins^2':            tm ** 2,
        'total_duration_mins journey_month': tm * journey_month,
        'journey_month^2':                  journey_month ** 2,
    }
    return pd.DataFrame([row])[MODEL_FEATURES]


def build_feature_matrix(combos: list) -> pd.DataFrame:
    """
    Build a single feature DataFrame for N predictions at once.
    WHY batch matrix over N separate DataFrames:
      RandomForest.predict() overhead is ~90ms regardless of N.
      N=1 → 90ms. N=288 → still ~117ms. Sequential = N×90ms = 26s.
      One call with N rows = N×faster at essentially zero extra cost.

    combos: list of dicts, each with keys:
      airline, source, destination, dep_hour, journey_month,
      journey_weekday, journey_day, duration_hours
    """
    rows = []
    for c in combos:
        tm   = c['duration_hours'] * 60.0
        dm   = (c['duration_hours'] % 1) * 60.0
        wday = c['journey_weekday']
        mon  = c['journey_month']
        rows.append({
            'journey_day':                      c['journey_day'],
            'journey_month':                    mon,
            'journey_weekday':                  wday,
            'is_weekend':                       1 if wday >= 5 else 0,
            'quarter':                          (mon - 1) // 3 + 1,
            'dep_hour':                         c['dep_hour'],
            'weekday':                          wday,
            'is_holiday':                       0,
            'duration_hours':                   c['duration_hours'],
            'duration_minutes':                 dm,
            'total_duration_mins':              tm,
            'Source_freq':                      SOURCE_FREQ.get(c['source'], 0.2),
            'Destination_freq':                 DEST_FREQ.get(c['destination'], 0.2),
            'Airline_mean_price':               AIRLINE_MEAN_PRICE.get(c['airline'], PRICE_AVG),
            'Source_mean_price':                SOURCE_MEAN_PRICE.get(c['source'], PRICE_AVG),
            'total_duration_mins.1':            tm,
            'journey_month.1':                  mon,
            'total_duration_mins^2':            tm ** 2,
            'total_duration_mins journey_month': tm * mon,
            'journey_month^2':                  mon ** 2,
        })
    return pd.DataFrame(rows)[MODEL_FEATURES]


def batch_predict(model, combos: list, passengers: int = 1,
                  fallback_fn=None) -> list:
    """
    Predict prices for N combos in ONE model.predict() call.

    WHY accept model as argument (not global):
      preprocessor.py has NO Streamlit dependency. The model object lives
      in app.py (loaded via @st.cache_resource). Passing it as an argument
      keeps this function pure and independently testable.

    Returns list of INR prices × passengers, same order as combos.
    Falls back to fallback_fn if model is None or prediction fails.
    """
    if model is not None and len(combos) > 0:
        try:
            X_batch = build_feature_matrix(combos)
            raws    = model.predict(X_batch)
            results = []
            for raw in raws:
                if raw < 15:
                    raw = float(np.expm1(raw))
                results.append(round(float(raw) * passengers, 2))
            return results
        except Exception:
            pass   # fall through to fallback

    if fallback_fn is not None:
        return [fallback_fn(c, passengers) for c in combos]

    # Last-resort: weighted dataset mean blend
    out = []
    for c in combos:
        base = (
            AIRLINE_MEAN.get(c['airline'], PRICE_AVG)                       * 0.30 +
            STOPS_MEAN.get(c.get('stops','non-stop'), PRICE_AVG)             * 0.25 +
            ROUTE_MEAN.get((c['source'], c['destination']), PRICE_AVG)       * 0.20 +
            HOUR_MEAN.get(c['dep_hour'], PRICE_AVG)                          * 0.10 +
            MONTH_MEAN.get(c['journey_month'], PRICE_AVG)                    * 0.10 +
            WEEKDAY_MEAN.get(c['journey_weekday'], PRICE_AVG)                * 0.05
        )
        out.append(round(base * (1 + max(0, c['duration_hours']-2)*0.04) * passengers, 2))
    return out



Overwriting /content/drive/MyDrive/AirFair-Vista/backend/preprocessor.py


---
##  Step: Environment Setup — Model Verification & Package Installation

**Why:** Before writing any application code, verifying the model file exists and has a non-zero size catches deployment errors early (corrupted serialisation, wrong path). Installing `streamlit`, `scikit-learn`, and `joblib` in the Colab environment ensures the app runtime matches the training environment — preventing the silent version mismatch failures that are the #1 cause of ML deployment bugs.

---
##  Step: Frontend — `app.py` (Streamlit User Interface)

**Why:** The Streamlit app is the product — it translates the ML model into a tool real users can interact with. The UI design follows the prediction workflow: (1) user selects flight attributes via dropdowns/sliders → (2) backend validates inputs → (3) `predict_price()` is called → (4) result displayed with Plotly price gauge and comparison charts. Plotly is used over matplotlib because Streamlit renders it natively as interactive HTML — users can hover, zoom, and explore the price visualisations without any page reload.

In [ ]:
%%writefile /content/drive/MyDrive/AirFair-Vista/app/app.py

import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os
import sys
import plotly.graph_objects as go
import plotly.express as px

# ── Import preprocessing pipeline ────────────────────────────────────────────

sys.path.insert(0, '/content/drive/MyDrive/AirFair-Vista/backend')
from preprocessor import (
    AIRLINES, SOURCES, DESTINATIONS, STOPS, MONTHS, WEEKDAYS,
    AIRLINE_MEAN_PRICE, SOURCE_FREQ, DEST_FREQ, SOURCE_MEAN_PRICE,
    PRICE_MIN, PRICE_MAX, PRICE_AVG, PRICE_MED,
    AIRLINE_MEAN, STOPS_MEAN, ROUTE_MEAN, HOUR_MEAN, MONTH_MEAN, WEEKDAY_MEAN,
    DURATION_LOOKUP, AVG_SPEED_BY_STOPS, CITY_COORDS,
    VALID_ROUTES, VALID_DESTINATIONS, VALID_AIRLINE_STOPS,
    MAX_PAX_BY_STOPS, INDIAN_HOLIDAYS,
    MODEL_PATH, MODEL_FEATURES,
    haversine_km, is_indian_holiday, predict_duration,
    get_validation_errors, build_features, build_feature_matrix,
)

# ── Model loading (cached singleton) ─────────────────────────────────────────
# WHY cache_resource not cache_data:
#   cache_resource → shared singleton (model, DB conn) across ALL users
#   cache_data     → per-argument cache for pure functions
#   Model is 53MB singleton → cache_resource correct choice.
@st.cache_resource(show_spinner='Loading model...')
def load_model():
    if os.path.exists(MODEL_PATH):
        pkl = joblib.load(MODEL_PATH)
        return pkl['model'], True
    return None, False

model, MODEL_LOADED = load_model()


def predict_price(airline, source, destination, stops,
                  dep_hour, journey_month, journey_weekday,
                  journey_day, duration_hours, passengers):
    """
    Single prediction wrapper — needs model object from this scope.
    Uses build_features() from preprocessor for feature engineering.
    For N predictions use batch_predict_app() (276x faster).
    """
    if MODEL_LOADED:
        try:
            X   = build_features(airline, source, destination,
                                  dep_hour, journey_month, journey_weekday,
                                  journey_day, duration_hours)
            raw = model.predict(X)[0]
            if raw < 15:
                raw = np.expm1(raw)
            return round(float(raw) * passengers, 2)
        except Exception as e:
            st.warning(f' Model error: {e}  →  fallback estimator.')
    base = (
        AIRLINE_MEAN.get(airline, PRICE_AVG)              * 0.30 +
        STOPS_MEAN.get(stops, PRICE_AVG)                  * 0.25 +
        ROUTE_MEAN.get((source, destination), PRICE_AVG)  * 0.20 +
        HOUR_MEAN.get(dep_hour, PRICE_AVG)                * 0.10 +
        MONTH_MEAN.get(journey_month, PRICE_AVG)          * 0.10 +
        WEEKDAY_MEAN.get(journey_weekday, PRICE_AVG)      * 0.05
    )
    return round(base * (1 + max(0, duration_hours-2)*0.04) * passengers, 2)


def batch_predict_app(combos: list, passengers: int = 1) -> list:
    """
    Batch predict N combos in ONE model.predict() call via preprocessor.
    WHY wrapper: preprocessor is model-agnostic; model lives here.
    Performance: 12 airlines batch = 110ms vs 12x sequential = 1360ms.
    """
    from preprocessor import batch_predict
    return batch_predict(
        model if MODEL_LOADED else None,
        combos, passengers
    )


st.set_page_config(
    page_title='AirFair Vista',
    page_icon='✈️',
    layout='wide',
    initial_sidebar_state='expanded'
)


# ─────────────────────────────────────────────────────────────────────────────
#  CSS
# ─────────────────────────────────────────────────────────────────────────────
st.markdown("""
<style>
/* ── FONTS ──────────────────────────────────────────────────────── */
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@300;400;500;600;700;800&family=Syne:wght@700;800;900&display=swap');

/* ── CSS VARIABLES ──────────────────────────────────────────────── */
:root {
    --primary:       #0052cc;
    --primary-dark:  #003a99;
    --primary-light: #e8f0fe;
    --accent:        #f5a623;
    --accent-light:  rgba(245,166,35,0.15);
    --success:       #22c55e;
    --danger:        #ef4444;
    --warning:       #f59e0b;
    --bg-main:       #f0f4ff;
    --bg-card:       #ffffff;
    --bg-sidebar:    #09122c;
    --text-primary:  #0f172a;
    --text-secondary:#64748b;
    --text-muted:    #94a3b8;
    --border:        #e2e8f0;
    --border-focus:  #0052cc;
    --shadow-sm:     0 1px 3px rgba(0,0,0,0.06), 0 1px 2px rgba(0,0,0,0.04);
    --shadow-md:     0 4px 16px rgba(0,0,0,0.08), 0 2px 6px rgba(0,0,0,0.05);
    --shadow-lg:     0 8px 32px rgba(0,0,0,0.10), 0 4px 12px rgba(0,0,0,0.06);
    --radius-sm:     8px;
    --radius-md:     12px;
    --radius-lg:     18px;
    --radius-xl:     24px;
}

/* ── GLOBAL ─────────────────────────────────────────────────────── */
html, body, [class*="css"] {
    font-family: 'Plus Jakarta Sans', sans-serif !important;
    font-feature-settings: 'cv02','cv03','cv04','cv11';
}
.stApp {
    background: var(--bg-main);
    background-image:
        radial-gradient(ellipse at 20% 0%, rgba(0,82,204,0.06) 0%, transparent 60%),
        radial-gradient(ellipse at 80% 100%, rgba(245,166,35,0.04) 0%, transparent 60%);
}
* { box-sizing: border-box; }

/* ── SIDEBAR ─────────────────────────────────────────────────────── */
[data-testid="stSidebar"] {
    background: var(--bg-sidebar) !important;
    border-right: 1px solid rgba(255,255,255,0.06) !important;
}
[data-testid="stSidebar"]::before {
    content: '';
    position: fixed;
    top: 0; left: 0;
    width: 280px; height: 100vh;
    background: linear-gradient(180deg,
        rgba(0,82,204,0.15) 0%,
        transparent 40%,
        rgba(245,166,35,0.05) 100%);
    pointer-events: none;
    z-index: 0;
}
[data-testid="stSidebar"] p,
[data-testid="stSidebar"] label,
[data-testid="stSidebar"] .stMarkdown,
[data-testid="stSidebar"] span { color: #cbd5e1 !important; }
[data-testid="stSidebar"] .stSelectbox label { color: #94a3b8 !important; font-size: 0.72rem !important; }

/* Sidebar brand */
.sb-brand {
    font-family: 'Syne', sans-serif;
    font-size: 1.45rem;
    font-weight: 900;
    color: #fff;
    letter-spacing: -0.5px;
    line-height: 1.1;
}
.sb-brand span { color: var(--accent); }
.sb-tagline {
    font-size: 0.65rem;
    color: #64748b;
    letter-spacing: 2px;
    text-transform: uppercase;
    margin-top: 2px;
}
.sb-divider {
    height: 1px;
    background: linear-gradient(90deg, transparent, rgba(255,255,255,0.08), transparent);
    margin: 16px 0;
}
.sb-section-label {
    font-size: 0.58rem !important;
    font-weight: 700 !important;
    letter-spacing: 2px !important;
    text-transform: uppercase !important;
    color: var(--accent) !important;
    margin: 20px 0 10px !important;
    display: flex;
    align-items: center;
    gap: 6px;
}
.sb-section-label::after {
    content: '';
    flex: 1;
    height: 1px;
    background: rgba(245,166,35,0.25);
}
.sb-info-row {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 6px 0;
    border-bottom: 1px solid rgba(255,255,255,0.04);
    font-size: 0.76rem;
}
.sb-info-row .k { color: #64748b; }
.sb-info-row .v { color: #e2e8f0; font-weight: 600; font-variant-numeric: tabular-nums; }
.sb-model-ok {
    background: rgba(34,197,94,0.12);
    border: 1px solid rgba(34,197,94,0.3);
    color: #4ade80;
    border-radius: var(--radius-sm);
    padding: 9px 12px;
    font-size: 0.76rem;
    font-weight: 600;
    margin-top: 10px;
    line-height: 1.5;
}
.sb-model-err {
    background: rgba(239,68,68,0.12);
    border: 1px solid rgba(239,68,68,0.3);
    color: #f87171;
    border-radius: var(--radius-sm);
    padding: 9px 12px;
    font-size: 0.76rem;
    font-weight: 600;
    margin-top: 10px;
}

/* ── PAGE HEADER ─────────────────────────────────────────────────── */
.page-header {
    background: linear-gradient(135deg,
        #09122c 0%, #0d1f5c 40%, #0052cc 100%);
    border-radius: var(--radius-xl);
    padding: 36px 40px 32px;
    margin-bottom: 28px;
    position: relative;
    overflow: hidden;
    box-shadow: 0 8px 32px rgba(0,82,204,0.25);
}
.page-header::before {
    content: '✈';
    position: absolute;
    right: 36px; top: 16px;
    font-size: 8rem;
    opacity: 0.05;
    transform: rotate(-20deg) scale(1.2);
    line-height: 1;
}
.page-header::after {
    content: '';
    position: absolute;
    bottom: -40px; left: -40px;
    width: 200px; height: 200px;
    border-radius: 50%;
    background: radial-gradient(circle, rgba(245,166,35,0.12) 0%, transparent 70%);
}
.page-header h1 {
    font-family: 'Syne', sans-serif;
    font-size: 2.1rem;
    font-weight: 900;
    margin: 0 0 6px;
    color: #fff;
    letter-spacing: -0.5px;
    line-height: 1.1;
}
.page-header h1 em { color: var(--accent); font-style: normal; }
.page-header p {
    color: rgba(255,255,255,0.55);
    font-size: 0.88rem;
    margin: 0;
    font-weight: 400;
}
.model-pill {
    display: inline-flex;
    align-items: center;
    gap: 5px;
    margin-top: 14px;
    background: rgba(34,197,94,0.15);
    border: 1px solid rgba(34,197,94,0.4);
    color: #4ade80;
    border-radius: 20px;
    padding: 4px 14px;
    font-size: 0.72rem;
    font-weight: 700;
    letter-spacing: 0.5px;
}
.model-pill-warn {
    display: inline-flex;
    align-items: center;
    gap: 5px;
    margin-top: 14px;
    background: rgba(245,158,11,0.15);
    border: 1px solid rgba(245,158,11,0.4);
    color: #fbbf24;
    border-radius: 20px;
    padding: 4px 14px;
    font-size: 0.72rem;
    font-weight: 700;
}

/* ── FORM CARD ───────────────────────────────────────────────────── */
[data-testid="stForm"] {
    background: var(--bg-card);
    border: 1px solid var(--border);
    border-radius: var(--radius-lg);
    padding: 28px 32px;
    box-shadow: var(--shadow-md);
}
.form-card {
    background: var(--bg-card);
    border: 1px solid var(--border);
    border-radius: var(--radius-lg);
    padding: 28px 32px 20px;
    box-shadow: var(--shadow-md);
    margin-bottom: 24px;
}
.form-section-title {
    font-size: 0.62rem;
    font-weight: 800;
    letter-spacing: 2px;
    text-transform: uppercase;
    color: var(--primary);
    margin-bottom: 16px;
    padding-bottom: 8px;
    border-bottom: 2px solid var(--primary-light);
    display: flex;
    align-items: center;
    gap: 6px;
}

/* ── WIDGETS ─────────────────────────────────────────────────────── */
.stSelectbox > div > div {
    background: #f8faff !important;
    border: 1.5px solid var(--border) !important;
    border-radius: var(--radius-sm) !important;
    transition: border-color 0.15s ease !important;
}
.stSelectbox > div > div:focus-within {
    border-color: var(--primary) !important;
    box-shadow: 0 0 0 3px rgba(0,82,204,0.1) !important;
}
.stSelectbox label {
    font-size: 0.78rem !important;
    font-weight: 600 !important;
    color: var(--text-secondary) !important;
    text-transform: uppercase !important;
    letter-spacing: 0.5px !important;
}
.stNumberInput > div > div > input {
    background: #f8faff !important;
    border: 1.5px solid var(--border) !important;
    border-radius: var(--radius-sm) !important;
    font-family: 'Plus Jakarta Sans', sans-serif !important;
    font-weight: 600 !important;
}
.stSlider label {
    font-size: 0.78rem !important;
    font-weight: 600 !important;
    color: var(--text-secondary) !important;
    text-transform: uppercase !important;
    letter-spacing: 0.5px !important;
}
.stSlider > div > div > div > div {
    background: var(--primary) !important;
}
[data-testid="stDateInput"] label {
    font-size: 0.78rem !important;
    font-weight: 600 !important;
    color: var(--text-secondary) !important;
    text-transform: uppercase !important;
    letter-spacing: 0.5px !important;
}
[data-testid="stDateInput"] > div > div > input {
    background: #f8faff !important;
    border: 1.5px solid var(--border) !important;
    border-radius: var(--radius-sm) !important;
    font-family: 'Plus Jakarta Sans', sans-serif !important;
    font-weight: 600 !important;
}

/* ── PREDICT BUTTON ─────────────────────────────────────────────── */
.stFormSubmitButton > button {
    background: linear-gradient(90deg, var(--primary), var(--primary-dark)) !important;
    color: white !important;
    font-family: 'Syne', sans-serif !important;
    font-weight: 700 !important;
    font-size: 1rem !important;
    border-radius: var(--radius-md) !important;
    border: none !important;
    padding: 15px 28px !important;
    letter-spacing: 0.3px;
    transition: all 0.2s ease !important;
    position: relative !important;
    overflow: hidden !important;
}
.stFormSubmitButton > button::before {
    content: '';
    position: absolute;
    top: 0; left: -100%;
    width: 100%; height: 100%;
    background: linear-gradient(90deg, transparent, rgba(255,255,255,0.1), transparent);
    transition: left 0.4s ease;
}
.stFormSubmitButton > button:hover::before { left: 100%; }
.stFormSubmitButton > button:hover {
    transform: translateY(-1px) !important;
    box-shadow: 0 8px 24px rgba(0,82,204,0.35) !important;
}
.stFormSubmitButton > button:disabled {
    background: #94a3b8 !important;
    cursor: not-allowed !important;
    transform: none !important;
    box-shadow: none !important;
}

/* ── RESULT TICKET ──────────────────────────────────────────────── */
.result-ticket {
    background: var(--bg-card);
    border-radius: var(--radius-xl);
    border: 1px solid var(--border);
    overflow: hidden;
    box-shadow: var(--shadow-lg);
    margin-bottom: 24px;
}
.ticket-header {
    background: linear-gradient(135deg, #09122c 0%, #0d1f5c 50%, #0052cc 100%);
    padding: 20px 28px;
    display: flex;
    justify-content: space-between;
    align-items: center;
    position: relative;
    overflow: hidden;
}
.ticket-header::after {
    content: '';
    position: absolute; top: 0; left: 0; right: 0;
    height: 2px;
    background: linear-gradient(90deg, var(--accent), #ff6b35, var(--accent));
}
.ticket-airline {
    font-family: 'Syne', sans-serif;
    color: white;
    font-size: 0.85rem;
    font-weight: 700;
    letter-spacing: 1px;
    text-transform: uppercase;
}
.ticket-date { color: rgba(255,255,255,0.5); font-size: 0.72rem; margin-top: 2px; }
.ticket-tag {
    background: rgba(245,166,35,0.18);
    border: 1px solid rgba(245,166,35,0.45);
    color: var(--accent);
    border-radius: 20px;
    padding: 4px 12px;
    font-size: 0.7rem;
    font-weight: 700;
    margin-left: 6px;
    letter-spacing: 0.3px;
}
.ticket-body {
    padding: 28px 32px;
    display: flex;
    align-items: center;
    gap: 0;
    background: white;
}
.ticket-city { text-align: center; min-width: 100px; flex-shrink: 0; }
.ticket-city .code {
    font-family: 'Syne', sans-serif;
    font-size: 2.6rem;
    font-weight: 900;
    color: var(--text-primary);
    line-height: 1;
    letter-spacing: -1px;
}
.ticket-city .name {
    font-size: 0.65rem;
    color: var(--text-muted);
    text-transform: uppercase;
    letter-spacing: 1.5px;
    margin-top: 5px;
    font-weight: 600;
}
.ticket-city .time {
    font-size: 0.88rem;
    color: var(--primary);
    font-weight: 700;
    margin-top: 8px;
    font-variant-numeric: tabular-nums;
}
.ticket-mid {
    flex: 1;
    text-align: center;
    padding: 0 24px;
    position: relative;
}
.ticket-mid .stops-label {
    font-size: 0.65rem;
    color: var(--accent);
    font-weight: 800;
    letter-spacing: 1px;
    text-transform: uppercase;
    margin-bottom: 8px;
    display: block;
}
.ticket-mid .dash-line {
    border-top: 2px dashed #cbd5e1;
    position: relative;
    margin: 0;
}
.ticket-mid .plane {
    position: absolute;
    top: -13px;
    left: 50%;
    transform: translateX(-50%);
    background: white;
    padding: 0 10px;
    font-size: 1.15rem;
    line-height: 1;
}
.ticket-mid .dur {
    font-size: 0.73rem;
    color: var(--text-muted);
    margin-top: 8px;
    line-height: 1.8;
    display: block;
}
.ticket-footer {
    border-top: 2px dashed #e2e8f0;
    padding: 22px 32px;
    display: flex;
    justify-content: space-between;
    align-items: center;
    background: #fafbff;
}
.price-label {
    font-size: 0.62rem;
    color: var(--text-muted);
    text-transform: uppercase;
    letter-spacing: 1.5px;
    margin-bottom: 6px;
    font-weight: 700;
}
.price-amount {
    font-family: 'Syne', sans-serif;
    font-size: 2.8rem;
    font-weight: 900;
    color: var(--primary);
    line-height: 1;
    letter-spacing: -1px;
}
.vs-avg-up   { color: var(--danger);  font-weight: 700; font-size: 0.8rem; margin-top: 5px; display: block; }
.vs-avg-down { color: var(--success); font-weight: 700; font-size: 0.8rem; margin-top: 5px; display: block; }
.route-info-block { text-align: center; }
.route-info-label {
    font-size: 0.58rem;
    color: var(--text-muted);
    text-transform: uppercase;
    letter-spacing: 1.5px;
    margin-bottom: 10px;
    font-weight: 700;
}
.route-info-items { display: flex; gap: 24px; justify-content: center; }
.route-info-item { text-align: center; }
.route-info-item .icon { font-size: 1.1rem; display: block; margin-bottom: 3px; }
.route-info-item .val {
    font-family: 'Syne', sans-serif;
    font-size: 0.88rem;
    font-weight: 800;
    color: var(--text-primary);
    display: block;
}
.route-info-item .sub {
    font-size: 0.6rem;
    color: var(--text-muted);
    text-transform: uppercase;
    letter-spacing: 1px;
    font-weight: 600;
}
.per-pax-block { text-align: right; }
.per-pax-label {
    font-size: 0.62rem;
    color: var(--text-muted);
    text-transform: uppercase;
    letter-spacing: 1px;
    font-weight: 700;
    margin-bottom: 4px;
}
.per-pax-val {
    font-family: 'Syne', sans-serif;
    font-size: 1.45rem;
    font-weight: 800;
    color: var(--text-primary);
}
.dataset-avg {
    font-size: 0.68rem;
    color: var(--text-muted);
    margin-top: 3px;
}

/* ── OUTPUT SECTION TITLES ──────────────────────────────────────── */
.output-section-title {
    font-family: 'Syne', sans-serif;
    font-size: 0.95rem;
    font-weight: 800;
    color: var(--text-primary);
    margin: 28px 0 14px;
    display: flex;
    align-items: center;
    gap: 8px;
    letter-spacing: -0.2px;
}
.output-section-title::after {
    content: '';
    flex: 1;
    height: 1.5px;
    background: linear-gradient(90deg, var(--primary-light), transparent);
    margin-left: 8px;
}

/* ── METRIC CARDS ────────────────────────────────────────────────── */
[data-testid="stMetric"] {
    background: var(--bg-card) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--radius-md) !important;
    padding: 16px 20px !important;
    box-shadow: var(--shadow-sm) !important;
    transition: box-shadow 0.2s ease !important;
}
[data-testid="stMetric"]:hover {
    box-shadow: var(--shadow-md) !important;
}
[data-testid="stMetricLabel"] > div {
    font-size: 0.7rem !important;
    font-weight: 700 !important;
    text-transform: uppercase !important;
    letter-spacing: 0.8px !important;
    color: var(--text-muted) !important;
}
[data-testid="stMetricValue"] > div {
    font-family: 'Syne', sans-serif !important;
    font-size: 1.5rem !important;
    font-weight: 800 !important;
    color: var(--text-primary) !important;
}

/* ── DATAFRAME ───────────────────────────────────────────────────── */
.stDataFrame {
    border-radius: var(--radius-md) !important;
    overflow: hidden !important;
    border: 1px solid var(--border) !important;
    box-shadow: var(--shadow-sm) !important;
}

/* ── ALERTS ──────────────────────────────────────────────────────── */
[data-testid="stAlert"] {
    border-radius: var(--radius-md) !important;
    font-size: 0.85rem !important;
    font-weight: 500 !important;
    border-left-width: 4px !important;
}

/* ── DIVIDER ─────────────────────────────────────────────────────── */
hr {
    border: none !important;
    border-top: 1px solid var(--border) !important;
    margin: 20px 0 !important;
}

/* ── DURATION INFO BOX ──────────────────────────────────────────── */
.duration-chip {
    display: inline-flex;
    align-items: center;
    gap: 8px;
    background: var(--primary-light);
    border: 1.5px solid rgba(0,82,204,0.2);
    border-radius: var(--radius-sm);
    padding: 10px 16px;
    font-size: 0.84rem;
    font-weight: 600;
    color: var(--primary);
    margin: 8px 0 4px;
    width: 100%;
}
.duration-chip .dur-val {
    font-family: 'Syne', sans-serif;
    font-size: 1rem;
    font-weight: 900;
    color: var(--primary-dark);
}

/* ── VALIDATION BADGES ──────────────────────────────────────────── */
.valid-badge {
    display: inline-flex;
    align-items: center;
    gap: 4px;
    background: rgba(34,197,94,0.1);
    border: 1px solid rgba(34,197,94,0.3);
    color: #16a34a;
    border-radius: 20px;
    padding: 2px 10px;
    font-size: 0.68rem;
    font-weight: 700;
    margin-top: 4px;
}

/* ── SCROLLBAR ───────────────────────────────────────────────────── */
::-webkit-scrollbar { width: 6px; height: 6px; }
::-webkit-scrollbar-track { background: transparent; }
::-webkit-scrollbar-thumb { background: #cbd5e1; border-radius: 3px; }
::-webkit-scrollbar-thumb:hover { background: #94a3b8; }
</style>
""", unsafe_allow_html=True)



# ─────────────────────────────────────────────────────────────────────────────
#  ① INTERACTIVE SIDEBAR
# ─────────────────────────────────────────────────────────────────────────────
with st.sidebar:

    st.markdown("""
    <div style="padding:8px 0 20px">
        <div class="sb-brand">✈ AirFair Vista</div>
        <div class="sb-tagline">Flight Price Intelligence</div>
    </div>
    """, unsafe_allow_html=True)

    st.divider()

    # Display settings
    st.markdown('<div class="sb-section-label">Display Settings</div>',
                unsafe_allow_html=True)
    currency       = st.selectbox('Currency',
                                  ['INR (₹)', 'USD ($)', 'EUR (€)', 'GBP (£)'])
    show_range     = st.toggle('Show Price Range',        value=True)
    show_compare   = st.toggle('Show Airline Comparison', value=True)

    st.divider()

    # Model status
    st.markdown('<div class="sb-section-label">Model Status</div>',
                unsafe_allow_html=True)
    if MODEL_LOADED:
        st.markdown(
            '<div class="sb-model-ok">✅ Pipeline loaded<br>'
            '<span style="font-weight:400;font-size:0.72rem;">'
            'flight_price_prediction_pipeline.pkl</span></div>',
            unsafe_allow_html=True
        )
    else:
        st.markdown(
            '<div class="sb-model-err">⚠️ Model not found<br>'
            '<span style="font-weight:400;font-size:0.72rem;">'
            'Using fallback estimator</span></div>',
            unsafe_allow_html=True
        )

    st.divider()

    # Dataset info
    st.markdown('<div class="sb-section-label">Dataset Info</div>',
                unsafe_allow_html=True)
    for k, v in [
        ('Records',   '10,682 flights'),
        ('Period',    'Mar – Jun 2019'),
        ('Routes',    '6 domestic'),
        ('Airlines',  '12 carriers'),
        ('Min price', '₹1,759'),
        ('Max price', '₹79,512'),
        ('Avg price', '₹9,087'),
    ]:
        st.markdown(
            f'<div class="sb-info-row">'
            f'<span class="k">{k}</span>'
            f'<span class="v">{v}</span></div>',
            unsafe_allow_html=True
        )


# Currency helpers
SYM = {'INR (₹)':'₹', 'USD ($)':'$', 'EUR (€)':'€', 'GBP (£)':'£'}
FAC = {'INR (₹)':1.0, 'USD ($)':0.012, 'EUR (€)':0.011, 'GBP (£)':0.0095}
sym = SYM[currency]
fac = FAC[currency]


# ═══════════════════════════════════════════════════════════════
#  PRICE PREDICTOR — ② Input Form  +  ③ Output Display
# ═══════════════════════════════════════════════════════════════
model_pill = (
    '<span class="model-pill">🤖 Real ML Pipeline Active</span>'
    if MODEL_LOADED
    else '<span class="model-pill-warn">⚠️ Fallback Estimator Active</span>'
)
st.markdown(f"""
<div class="page-header">
    <h1>Flight Price <em>Predictor</em></h1>
    <p>Indian domestic routes · 10,682 real flights · Mar–Jun 2019</p>
    {model_pill}
</div>
""", unsafe_allow_html=True)

# ── ② SMART INPUT FORM (no st.form — uses session_state for live updates) ───
import datetime as _dt

# ── Session state initialisation ────────────────────────────────────────────
if 'source' not in st.session_state:
    st.session_state.source      = SOURCES[0]
if 'destination' not in st.session_state:
    st.session_state.destination = VALID_DESTINATIONS[SOURCES[0]][0]
if 'airline' not in st.session_state:
    st.session_state.airline     = AIRLINES[0]
if 'stops' not in st.session_state:
    st.session_state.stops       = VALID_AIRLINE_STOPS[AIRLINES[0]][0]
if 'submitted' not in st.session_state:
    st.session_state.submitted   = False

# ── Callbacks for smart auto-update ─────────────────────────────────────────
def on_source_change():
    """When source changes → auto-set destination to first valid option."""
    new_src  = st.session_state['_source_sel']
    st.session_state.source      = new_src
    valid    = VALID_DESTINATIONS.get(new_src, DESTINATIONS)
    # Reset destination to first valid option for new source
    st.session_state.destination = valid[0]
    st.session_state['_dst_sel']  = valid[0]
    st.session_state.submitted    = False

def on_destination_change():
    st.session_state.destination = st.session_state['_dst_sel']
    st.session_state.submitted   = False

def on_airline_change():
    """When airline changes → auto-set stops to first valid option for that airline."""
    new_al   = st.session_state['_airline_sel']
    st.session_state.airline     = new_al
    valid    = VALID_AIRLINE_STOPS.get(new_al, STOPS)
    # Reset stops to first valid option for new airline
    st.session_state.stops       = valid[0]
    st.session_state['_stops_sel'] = valid[0]
    st.session_state.submitted    = False

def on_stops_change():
    st.session_state.stops     = st.session_state['_stops_sel']
    st.session_state.submitted = False

# ── Form card wrapper ────────────────────────────────────────────────────────
st.markdown('<div class="form-card">', unsafe_allow_html=True)
with st.container():

    # Section A — Route
    st.markdown('<div class="form-section-title"> Section A · Route</div>',
                unsafe_allow_html=True)
    col_src, col_arrow, col_dst = st.columns([5, 1, 5])

    # Source: triggers destination auto-update
    source = col_src.selectbox(
        'Source City', SOURCES,
        index=SOURCES.index(st.session_state.source),
        key='_source_sel',
        on_change=on_source_change
    )
    col_arrow.markdown(
        '<div style="text-align:center;padding-top:28px;'
        'font-size:1.3rem;color:#0f4c9a">→</div>',
        unsafe_allow_html=True
    )
    # Destination: auto-filtered AND auto-set from source change
    valid_dsts  = VALID_DESTINATIONS.get(st.session_state.source, DESTINATIONS)
    # Make sure stored destination is still valid for current source
    if st.session_state.destination not in valid_dsts:
        st.session_state.destination = valid_dsts[0]
    destination = col_dst.selectbox(
        'Destination City', valid_dsts,
        index=valid_dsts.index(st.session_state.destination),
        key='_dst_sel',
        on_change=on_destination_change
    )
    auto_dst = len(valid_dsts) == 1
    col_dst.markdown(
        f'<span class="valid-badge">'
        f'{" Auto-set" if auto_dst else "✅"} '
        f'{len(valid_dsts)} route(s) from {source}</span>',
        unsafe_allow_html=True
    )

    st.markdown('<br>', unsafe_allow_html=True)

    # Section B — Flight Details
    st.markdown('<div class="form-section-title"> Section B · Flight Details</div>',
                unsafe_allow_html=True)
    col_al, col_st = st.columns(2)

    # Airline: triggers stops auto-update
    airline = col_al.selectbox(
        'Airline', AIRLINES,
        index=AIRLINES.index(st.session_state.airline),
        key='_airline_sel',
        on_change=on_airline_change
    )
    # Stops: auto-filtered AND auto-set from airline change
    valid_stops = VALID_AIRLINE_STOPS.get(st.session_state.airline, STOPS)
    if st.session_state.stops not in valid_stops:
        st.session_state.stops = valid_stops[0]
    stops = col_st.selectbox(
        'Stops', valid_stops,
        index=valid_stops.index(st.session_state.stops),
        key='_stops_sel',
        on_change=on_stops_change
    )
    auto_stops = (st.session_state.stops == valid_stops[0])
    col_st.markdown(
        f'<span class="valid-badge">'
        f'{" Auto-set" if len(valid_stops)==1 else "✅"} '
        f'{len(valid_stops)} stop option(s) for {airline}</span>',
        unsafe_allow_html=True
    )

    # Duration auto-predicted from route + stops
    duration_hrs = predict_duration(destination, destination, stops) \
        if source == destination \
        else predict_duration(source, destination, stops)
    st.markdown(
        f'<div class="duration-chip">'
        f' Auto-predicted Duration: '
        f'<span class="dur-val">{duration_hrs:.1f}h</span>'
        f' &nbsp;·&nbsp; Based on real {source} → {destination} / {stops} records'
        f'</div>',
        unsafe_allow_html=True
    )

    st.markdown('<br>', unsafe_allow_html=True)

    # Section C — Journey Details
    st.markdown('<div class="form-section-title"> Section C · Journey Details</div>',
                unsafe_allow_html=True)
    cal1, cal2, cal3 = st.columns([3, 2, 1])
    travel_date = cal1.date_input(
        '📆 Travel Date',
        value=_dt.date(2019, 4, 15),
        min_value=_dt.date(2019, 3, 1),
        max_value=_dt.date(2019, 6, 30),
        help='Dataset: Mar–Jun 2019. Month, Day & Weekday are auto-derived.',
        key='_travel_date'
    )
    journey_month   = travel_date.month
    journey_day     = travel_date.day
    journey_weekday = travel_date.weekday()
    cal1.caption(
        f'📅 {travel_date.strftime("%A, %d %B %Y")} · '
        f'Month={journey_month}, Day={journey_day}, Weekday={journey_weekday}'
    )
    dep_hour   = cal2.slider(' Departure Hour', 0, 23, 8, key='_dep_hour')
    passengers = cal3.number_input('👥 Passengers', 1, 9, 1, key='_passengers')

    st.markdown('<br>', unsafe_allow_html=True)

    # Live validation feedback
    live_errors, live_warnings = get_validation_errors(
        source, destination, airline, stops, int(passengers), dep_hour
    )
    if live_errors:
        for e in live_errors:
            st.error(f' {e}')
    elif live_warnings:
        for w in live_warnings:
            st.warning(f' {w}')
    else:
        st.success('✅ All inputs valid — ready to predict!')

    # ── Task 2: Real-time live price preview ─────────────────────────────
    # Why: users can see price update instantly as they adjust inputs,
    # without waiting to press Predict. Uses same model pipeline — not fake.
    # Why NOT st.form: forms batch all inputs, blocking live updates.
    # We use regular widgets + session_state which reruns on every change.
    if not live_errors:
        _live_dur = predict_duration(source, destination, stops)
        _live_price = predict_price(
            airline, source, destination, stops,
            dep_hour, journey_month, journey_weekday,
            int(journey_day), _live_dur, int(passengers)
        ) * fac
        _live_avg = PRICE_AVG * fac
        _live_diff = _live_price - _live_avg
        _live_icon = '▲' if _live_diff > 0 else '▼'
        _live_col  = '#ef4444' if _live_diff > 0 else '#22c55e'
        st.markdown(
            f'<div style="background:linear-gradient(90deg,#eff6ff,#dbeafe);'
            f'border:1.5px solid #bfdbfe;border-radius:10px;'
            f'padding:12px 20px;display:flex;align-items:center;gap:16px;margin-bottom:4px">'
            f'<div style="font-size:0.65rem;font-weight:800;color:#3b82f6;'
            f'text-transform:uppercase;letter-spacing:1.5px">⚡ Live Estimate</div>'
            f'<div style="font-family:Syne,sans-serif;font-size:1.6rem;'
            f'font-weight:900;color:#0052cc;letter-spacing:-1px">{sym}{_live_price:,.0f}</div>'
            f'<div style="font-size:0.78rem;font-weight:600;color:{_live_col}">'
            f'{_live_icon} {sym}{abs(_live_diff):,.0f} vs dataset avg</div>'
            f'<div style="font-size:0.72rem;color:#64748b;margin-left:auto">'
            f'Updates live · {airline} · {source}→{destination} · {stops}'
            f'</div></div>',
            unsafe_allow_html=True
        )

    st.markdown('<br>', unsafe_allow_html=True)
    # ── Predict button ────────────────────────────────────────────────────
    btn_label = '🔍  Predict Price' if not live_errors else '⚠️  Fix Errors Above to Predict'
    if st.button(
        btn_label,
        use_container_width=True,
        type='primary',
        disabled=bool(live_errors),
        key='_predict_btn'
    ):
        st.session_state.submitted = False   # reset first to force rerun
        st.session_state.submitted   = True
        st.session_state._src_snap   = source
        st.session_state._dst_snap   = destination
        st.session_state._al_snap    = airline
        st.session_state._st_snap    = stops
        st.session_state._dur_snap   = duration_hrs
        st.session_state._hr_snap    = dep_hour
        st.session_state._mo_snap    = journey_month
        st.session_state._wd_snap    = journey_weekday
        st.session_state._dy_snap    = journey_day
        st.session_state._px_snap    = passengers
        st.session_state._td_snap    = travel_date
        st.rerun()   # immediately rerun to show output below

st.markdown('</div>', unsafe_allow_html=True)  # close form-card

# Snapshot values used for output (survive reruns)
submitted = st.session_state.submitted
if submitted:
    source          = st.session_state._src_snap
    destination     = st.session_state._dst_snap
    airline         = st.session_state._al_snap
    stops           = st.session_state._st_snap
    duration_hrs    = st.session_state._dur_snap
    dep_hour        = st.session_state._hr_snap
    journey_month   = st.session_state._mo_snap
    journey_weekday = st.session_state._wd_snap
    journey_day     = st.session_state._dy_snap
    passengers      = st.session_state._px_snap
    travel_date     = st.session_state._td_snap



if submitted:
    # ── ③ OUTPUT DISPLAY ─────────────────────────────────────────────────────

    # ── Advanced Input Validation ────────────────────────────────────────
    errors, warnings = get_validation_errors(
        source, destination, airline, stops,
        int(passengers), dep_hour
    )

    # Show all warnings first (non-blocking)
    for w in warnings:
        st.warning(f'⚠️ {w}')

    # Block on any errors
    if errors:
        for e in errors:
            st.error(f'❌ {e}')
        st.info(
            '💡 **Valid routes in this dataset:** '
            'Banglore→Delhi/New Delhi · Chennai→Kolkata · '
            'Delhi→Cochin · Kolkata→Banglore · Mumbai→Hyderabad'
        )
        st.stop()

    # ── Compute prediction (spinner shows while RF model runs) ─────────
    with st.spinner('🤖  Running ML model...'):
        price_inr = predict_price(
            airline, source, destination, stops,
            dep_hour, journey_month, journey_weekday,
            int(journey_day), duration_hrs, int(passengers)
        )
    price_d = price_inr * fac
    low_d   = price_d * 0.90
    high_d  = price_d * 1.15
    avg_d   = PRICE_AVG * fac
    diff    = price_d - avg_d

    src_code = source[:3].upper()
    dst_code = destination[:3].upper()
    dep_str  = f"{dep_hour:02d}:00"
    day_str  = f"{WEEKDAYS[journey_weekday][:3]}, {int(journey_day)} {MONTHS[journey_month]}"
    vs_class = 'vs-avg-up' if diff > 0 else 'vs-avg-down'
    vs_text  = (f'▲ {sym}{abs(diff):,.0f} above avg'
                if diff > 0 else f'▼ {sym}{abs(diff):,.0f} below avg')
    # Route distance — computed via Haversine formula from real GPS coordinates
    dist_km      = haversine_km(source, destination)
    dist_str     = f'{dist_km:,} km' if dist_km > 0 else 'N/A'
    speed_str    = f'{round(dist_km / duration_hrs):,} km/h' if dist_km > 0 and duration_hrs > 0 else 'N/A'
    price_per_km = f'{sym}{round(price_d / dist_km, 1)}/km' if dist_km > 0 else 'N/A'
    # Holiday indicator — real Indian public holiday check
    holiday_name = INDIAN_HOLIDAYS.get((journey_month, int(journey_day)), None)
    holiday_tag  = f'🎉 {holiday_name}' if holiday_name else ''

    # Reset button — lets user go back and change inputs
    rc1, rc2 = st.columns([6, 1])
    rc1.success('✅  Prediction ready!')
    if rc2.button('🔄 New', key='_reset_btn', help='Reset and make a new prediction'):
        st.session_state.submitted = False
        st.rerun()

    # Output A — Ticket Card
    st.markdown(
        '<div class="output-section-title">📋 Output A · Your Flight Estimate</div>',
        unsafe_allow_html=True
    )
    st.markdown(f"""
    <div class="result-ticket">
        <div class="ticket-header">
            <div>
                <div class="ticket-airline">{airline.upper()}</div>
                <div style="color:#aac4ef;font-size:0.72rem;margin-top:2px">{day_str} {holiday_tag}</div>
            </div>
            <div>
                <span class="ticket-tag">{stops}</span>
                <span class="ticket-tag">{duration_hrs}h</span>
                <span class="ticket-tag">{int(passengers)} pax</span>
                <span class="ticket-tag">{'🤖 ML Model' if MODEL_LOADED else '📊 Estimator'}</span>
            </div>
        </div>
        <div class="ticket-body">
            <div class="ticket-city">
                <div class="code">{src_code}</div>
                <div class="name">{source}</div>
                <div class="time">{dep_str}</div>
            </div>
            <div class="ticket-mid">
                <div class="stops-label">{stops.upper()}</div>
                <div class="dash-line"><span class="plane">✈️</span></div>
                <div class="dur">
                    <span style="color:#1e2b4a;font-weight:600">{duration_hrs}h</span>
                    &nbsp;·&nbsp;
                    <span style="color:#0f4c9a;font-weight:600">{dist_str}</span>
                </div>
            </div>
            <div class="ticket-city">
                <div class="code">{dst_code}</div>
                <div class="name">{destination}</div>
            </div>
        </div>
        <div class="ticket-footer">
            <div>
                <div class="price-label">Total Estimated Fare · {int(passengers)} Pax</div>
                <div class="price-amount">{sym}{price_d:,.0f}</div>
                <span class="{vs_class}">{vs_text}</span>
            </div>
            <div class="route-info-block">
                <div class="route-info-label">Route Info</div>
                <div class="route-info-items">
                    <div class="route-info-item">
                        <span class="icon">📍</span>
                        <span class="val">{dist_str}</span>
                        <span class="sub">Distance</span>
                    </div>
                    <div class="route-info-item">
                        <span class="icon">⚡</span>
                        <span class="val">{speed_str}</span>
                        <span class="sub">Avg Speed</span>
                    </div>
                    <div class="route-info-item">
                        <span class="icon">💰</span>
                        <span class="val" style="color:var(--primary)">{price_per_km}</span>
                        <span class="sub">Cost/km</span>
                    </div>
                </div>
            </div>
            <div class="per-pax-block">
                <div class="per-pax-label">Per Passenger</div>
                <div class="per-pax-val">{sym}{price_d / passengers:,.0f}</div>
                <div class="dataset-avg">Dataset avg: {sym}{avg_d:,.0f}</div>
            </div>
        </div>
    </div>
    """, unsafe_allow_html=True)

    # Output B — Price Range
    if show_range:
        st.markdown(
            '<div class="output-section-title">📉 Output B · Price Range</div>',
            unsafe_allow_html=True
        )
        b1, b2, b3, b4 = st.columns(4)
        b1.metric('🟢 Low Estimate',  f'{sym}{low_d:,.0f}',
                  delta=f'-{sym}{price_d - low_d:,.0f}')
        b2.metric('🎯 Predicted',     f'{sym}{price_d:,.0f}')
        b3.metric('🔴 High Estimate', f'{sym}{high_d:,.0f}',
                  delta=f'+{sym}{high_d - price_d:,.0f}')
        b4.metric('📊 Dataset Avg',   f'{sym}{avg_d:,.0f}')

    # ═══════════════════════════════════════════════════════════════
    #  OUTPUT C — PRICE COMPARISON VISUALIZATIONS (Task 4)
    #  All predictions are live model.predict() calls — nothing static
    # ═══════════════════════════════════════════════════════════════
    st.markdown(
        '<div class="output-section-title">📊 Output C · Price Comparison Visualizations</div>',
        unsafe_allow_html=True
    )

    # ── Shared Plotly theme ───────────────────────────────────────
    PLOT_BG    = '#ffffff'
    GRID_COLOR = '#e8f0fe'
    FONT_FMLY  = 'Plus Jakarta Sans, sans-serif'
    PRIMARY    = '#0052cc'
    ACCENT     = '#f5a623'
    SUCCESS    = '#22c55e'
    DANGER     = '#ef4444'
    PALETTE    = [
        '#0052cc','#f5a623','#22c55e','#ef4444','#8b5cf6',
        '#06b6d4','#f97316','#14b8a6','#e11d48','#64748b',
        '#a855f7','#10b981'
    ]

    def base_layout(title, xaxis_title='', yaxis_title=''):
        return dict(
            title=dict(
                text=title,
                font=dict(family=FONT_FMLY, size=14, color='#0f172a'),
                x=0, xanchor='left', pad=dict(l=4, b=12)
            ),
            paper_bgcolor=PLOT_BG, plot_bgcolor=PLOT_BG,
            font=dict(family=FONT_FMLY, color='#64748b', size=11),
            xaxis=dict(
                title=xaxis_title, gridcolor=GRID_COLOR,
                linecolor='#e2e8f0', tickfont=dict(size=10)
            ),
            yaxis=dict(
                title=yaxis_title, gridcolor=GRID_COLOR,
                linecolor='#e2e8f0', tickfont=dict(size=10),
                tickprefix=sym
            ),
            margin=dict(l=10, r=10, t=46, b=10),
            hoverlabel=dict(
                bgcolor='#0f172a', font_size=12,
                font_family=FONT_FMLY, font_color='white',
                bordercolor='#1e293b'
            ),
            showlegend=False
        )

    # ── Task 1: Batch airline predictions (12 rows → 1 call, 12× faster) ───────
    _base = dict(source=source, destination=destination,
                 dep_hour=dep_hour, journey_month=journey_month,
                 journey_weekday=journey_weekday, journey_day=int(journey_day),
                 duration_hours=duration_hrs)
    _al_combos = [{**_base, 'airline': al} for al in AIRLINES]
    _al_raw    = batch_predict_app(_al_combos, int(passengers))
    al_prices  = {al: round(p * fac, 2) for al, p in zip(AIRLINES, _al_raw)}

    al_df = (
        pd.DataFrame({'Airline': list(al_prices.keys()),
                      'Price':   list(al_prices.values())})
        .sort_values('Price')
        .reset_index(drop=True)
    )
    al_df['Color']    = [SUCCESS if v < price_d else (DANGER if v > price_d else ACCENT)
                         for v in al_df['Price']]
    al_df['Selected'] = al_df['Airline'] == airline
    al_df['Label']    = al_df['Price'].apply(lambda v: f'{sym}{v:,.0f}')

    # ── Chart 1 + Chart 2 ─────────────────────────────────────────
    viz_c1, viz_c2 = st.columns(2)

    # Chart 1 — Horizontal bar: all airlines ranked
    with viz_c1:
        st.markdown(
            '<div style="font-size:0.72rem;font-weight:700;color:#64748b;'
            'text-transform:uppercase;letter-spacing:1px;margin-bottom:8px">'
            '🏷️ All Airlines · Ranked by Price</div>',
            unsafe_allow_html=True
        )
        bar_colors = [
            PRIMARY if a == airline else (
                '#e8f0fe' if al_prices[a] < price_d else '#fee2e2'
            )
            for a in al_df['Airline']
        ]
        text_colors = [
            'white' if a == airline else '#0f172a'
            for a in al_df['Airline']
        ]
        fig1 = go.Figure(go.Bar(
            x=al_df['Price'],
            y=al_df['Airline'],
            orientation='h',
            marker=dict(color=bar_colors, line=dict(width=0)),
            text=al_df['Label'],
            textposition='outside',
            textfont=dict(size=10, family=FONT_FMLY, color='#0f172a'),
            hovertemplate='<b>%{y}</b><br>Price: ' + sym + '%{x:,.0f}<extra></extra>',
            width=0.65
        ))
        # Selected airline marker line
        fig1.add_vline(
            x=price_d, line_width=2, line_dash='dash',
            line_color=ACCENT,
            annotation_text='Your pick',
            annotation_font=dict(size=10, color=ACCENT, family=FONT_FMLY),
            annotation_position='top right'
        )
        layout1 = base_layout('', xaxis_title=f'Price ({sym})')
        layout1['yaxis']['title'] = ''
        layout1['margin'] = dict(l=10, r=80, t=10, b=10)
        layout1['height'] = 340
        fig1.update_layout(**layout1)
        st.plotly_chart(fig1, use_container_width=True, config={'displayModeBar': False})

    # Chart 2 — Grouped bar: by stops for selected airline vs cheapest
    with viz_c2:
        st.markdown(
            '<div style="font-size:0.72rem;font-weight:700;color:#64748b;'
            'text-transform:uppercase;letter-spacing:1px;margin-bottom:8px">'
            '🔄 Price by Number of Stops</div>',
            unsafe_allow_html=True
        )
        stops_data   = {}
        cheapest_al  = al_df.iloc[0]['Airline']
        for st_opt in STOPS:
            stops_data[st_opt] = {
                'selected': predict_price(
                    airline, source, destination, st_opt,
                    dep_hour, journey_month, journey_weekday,
                    int(journey_day), duration_hrs, int(passengers)
                ) * fac,
                'cheapest': predict_price(
                    cheapest_al, source, destination, st_opt,
                    dep_hour, journey_month, journey_weekday,
                    int(journey_day), duration_hrs, int(passengers)
                ) * fac,
            }
        fig2 = go.Figure()
        fig2.add_trace(go.Bar(
            name=airline,
            x=STOPS,
            y=[stops_data[s]['selected'] for s in STOPS],
            marker_color=PRIMARY,
            text=[f'{sym}{stops_data[s]["selected"]:,.0f}' for s in STOPS],
            textposition='outside',
            textfont=dict(size=9, family=FONT_FMLY),
            hovertemplate='<b>' + airline + '</b><br>%{x}<br>' +
                          sym + '%{y:,.0f}<extra></extra>',
        ))
        if cheapest_al != airline:
            fig2.add_trace(go.Bar(
                name=cheapest_al,
                x=STOPS,
                y=[stops_data[s]['cheapest'] for s in STOPS],
                marker_color='#e8f0fe',
                text=[f'{sym}{stops_data[s]["cheapest"]:,.0f}' for s in STOPS],
                textposition='outside',
                textfont=dict(size=9, family=FONT_FMLY, color='#64748b'),
                hovertemplate='<b>' + cheapest_al + '</b><br>%{x}<br>' +
                              sym + '%{y:,.0f}<extra></extra>',
            ))
        layout2 = base_layout('', xaxis_title='Stops', yaxis_title=f'Price ({sym})')
        layout2['barmode']    = 'group'
        layout2['height']     = 340
        layout2['showlegend'] = True
        layout2['legend']     = dict(
            orientation='h', yanchor='bottom', y=1.02,
            xanchor='left', x=0,
            font=dict(size=10, family=FONT_FMLY)
        )
        layout2['margin'] = dict(l=10, r=10, t=40, b=10)
        fig2.update_layout(**layout2)
        st.plotly_chart(fig2, use_container_width=True, config={'displayModeBar': False})

    # ── Chart 3 + Chart 4 ─────────────────────────────────────────
    viz_c3, viz_c4 = st.columns(2)

    # Chart 3 — Line: price by departure hour for top 3 airlines
    with viz_c3:
        st.markdown(
            '<div style="font-size:0.72rem;font-weight:700;color:#64748b;'
            'text-transform:uppercase;letter-spacing:1px;margin-bottom:8px">'
            '⏰ Price vs Departure Hour (Top 3 Airlines)</div>',
            unsafe_allow_html=True
        )
        top3 = al_df.head(1)['Airline'].tolist() + [airline]
        if cheapest_al in top3: top3 = list(dict.fromkeys(top3))
        # Always include: cheapest, selected, and mid-range
        mid_idx  = len(al_df) // 2
        mid_al   = al_df.iloc[mid_idx]['Airline']
        top3     = list(dict.fromkeys([cheapest_al, airline, mid_al]))[:3]
        hours    = list(range(0, 24))
        fig3     = go.Figure()
        line_colors = [SUCCESS, PRIMARY, ACCENT]
        # Task 1: batch all (3 airlines × 24 hours = 72 rows) in one call
        _c3_combos = [{**_base, 'airline': al, 'dep_hour': h}
                      for al in top3 for h in hours]
        _c3_raw    = batch_predict_app(_c3_combos, int(passengers))
        _c3_prices = {al: [round(_c3_raw[ai*24+hi]*fac,2) for hi in range(24)]
                      for ai, al in enumerate(top3)}
        for idx, al in enumerate(top3):
            h_prices    = _c3_prices[al]
            is_selected = (al == airline)
            fig3.add_trace(go.Scatter(
                x=hours,
                y=h_prices,
                name=al,
                mode='lines+markers',
                line=dict(
                    color=line_colors[idx],
                    width=2.5 if is_selected else 1.5,
                    dash='solid' if is_selected else 'dot'
                ),
                marker=dict(size=5 if is_selected else 3),
                hovertemplate='<b>' + al + '</b><br>Hour: %{x}<br>Price: ' +
                              sym + '%{y:,.0f}<extra></extra>',
            ))
        # Mark the selected departure hour
        fig3.add_vline(
            x=dep_hour,
            line_width=1.5, line_dash='dash', line_color='#94a3b8',
            annotation_text=f'{dep_hour:02d}:00',
            annotation_font=dict(size=9, family=FONT_FMLY, color='#94a3b8'),
            annotation_position='top'
        )
        layout3 = base_layout('', xaxis_title='Hour', yaxis_title=f'Price ({sym})')
        layout3['height']     = 300
        layout3['showlegend'] = True
        layout3['legend']     = dict(
            orientation='h', yanchor='bottom', y=1.02,
            xanchor='left', x=0,
            font=dict(size=9, family=FONT_FMLY)
        )
        layout3['xaxis']['tickangle'] = 0
        layout3['xaxis']['nticks']    = 24
        layout3['xaxis']['tickformat'] = '02d'
        layout3['xaxis']['ticksuffix'] = ':00'
        layout3['margin'] = dict(l=10, r=10, t=40, b=40)
        fig3.update_layout(**layout3)
        st.plotly_chart(fig3, use_container_width=True, config={'displayModeBar': False})

    # Chart 4 — Scatter: price vs duration for all airlines
    with viz_c4:
        st.markdown(
            '<div style="font-size:0.72rem;font-weight:700;color:#64748b;'
            'text-transform:uppercase;letter-spacing:1px;margin-bottom:8px">'
            '⏱️ Price vs Duration (All Airlines)</div>',
            unsafe_allow_html=True
        )
        # Task 1: batch all 60 scatter combos (12 airlines × 5 stops) in one call
        _c4_meta   = [(al, st_opt, predict_duration(source, destination, st_opt))
                      for al in AIRLINES for st_opt in STOPS]
        _c4_combos = [{**_base, 'airline': al, 'duration_hours': dur}
                      for al, st_opt, dur in _c4_meta]
        _c4_raw    = batch_predict_app(_c4_combos, int(passengers))
        scatter_rows = []
        for (al, st_opt, dur), p_raw in zip(_c4_meta, _c4_raw):
            scatter_rows.append({
                'Airline':  al,
                'Duration': dur,
                'Price':    round(p_raw * fac, 2),
                'Stops':    st_opt,
                'Selected': (al == airline and st_opt == stops),
            })
        sc_df = pd.DataFrame(scatter_rows)

        fig4 = go.Figure()
        # All non-selected points
        mask_other    = ~sc_df['Selected']
        fig4.add_trace(go.Scatter(
            x=sc_df[mask_other]['Duration'],
            y=sc_df[mask_other]['Price'],
            mode='markers',
            name='Other options',
            marker=dict(
                color='#e8f0fe', size=8,
                line=dict(color='#94a3b8', width=1)
            ),
            customdata=sc_df[mask_other][['Airline','Stops']].values,
            hovertemplate=(
                '<b>%{customdata[0]}</b><br>'
                'Stops: %{customdata[1]}<br>'
                'Duration: %{x:.1f}h<br>'
                'Price: ' + sym + '%{y:,.0f}<extra></extra>'
            ),
        ))
        # Selected point (highlighted)
        mask_sel = sc_df['Selected']
        if mask_sel.any():
            fig4.add_trace(go.Scatter(
                x=sc_df[mask_sel]['Duration'],
                y=sc_df[mask_sel]['Price'],
                mode='markers+text',
                name='Your pick',
                marker=dict(
                    color=PRIMARY, size=14,
                    line=dict(color='white', width=2),
                    symbol='star'
                ),
                text=[f'  {airline}<br>  {sym}{price_d:,.0f}'],
                textposition='middle right',
                textfont=dict(size=10, color=PRIMARY, family=FONT_FMLY),
                hovertemplate=(
                    '<b>⭐ YOUR PICK</b><br>'
                    f'{airline}<br>'
                    'Duration: %{x:.1f}h<br>'
                    'Price: ' + sym + '%{y:,.0f}<extra></extra>'
                ),
            ))
        # Price average line
        fig4.add_hline(
            y=avg_d, line_width=1.5,
            line_dash='dash', line_color=ACCENT,
            annotation_text=f'Dataset avg {sym}{avg_d:,.0f}',
            annotation_font=dict(size=9, color=ACCENT, family=FONT_FMLY),
            annotation_position='bottom right'
        )
        layout4 = base_layout('',
                               xaxis_title='Duration (hours)',
                               yaxis_title=f'Price ({sym})')
        layout4['height']     = 300
        layout4['showlegend'] = True
        layout4['legend']     = dict(
            orientation='h', yanchor='bottom', y=1.02,
            xanchor='left', x=0,
            font=dict(size=9, family=FONT_FMLY)
        )
        layout4['margin'] = dict(l=10, r=10, t=40, b=10)
        fig4.update_layout(**layout4)
        st.plotly_chart(fig4, use_container_width=True, config={'displayModeBar': False})

    # ── Chart 5 — Full comparison table with visual indicators ───
    st.markdown(
        '<div class="output-section-title">📋 Output D · Full Price Comparison Table</div>',
        unsafe_allow_html=True
    )
    st.caption(
        f'All {len(AIRLINES)} airlines · {source} → {destination} · '
        f'{stops} · {travel_date.strftime("%d %b %Y")} · '
        f'{dep_hour:02d}:00 · {int(passengers)} pax · Sorted cheapest first.'
    )
    table_rows = []
    for al in AIRLINES:
        p       = al_prices[al]
        dur     = predict_duration(source, destination, stops)
        diff_v  = p - price_d
        saving  = price_d - p
        table_rows.append({
            'Airline':              al,
            f'Predicted ({sym})':  f'{sym}{p:,.0f}',
            'vs Your Pick':        (
                f'🟢 Save {sym}{saving:,.0f}'  if saving >  50
                else (f'🔴 +{sym}{abs(diff_v):,.0f}' if diff_v > 50
                      else '🟡 Same')
            ),
            'vs Dataset Avg':      (
                f'✅ {sym}{avg_d-p:,.0f} below avg' if p < avg_d
                else f'⚠️ {sym}{p-avg_d:,.0f} above avg'
            ),
            'Duration':            f'{dur:.1f}h',
            'Cost/km':             (
                f'{sym}{p/dist_km:.1f}/km'
                if dist_km > 0 else 'N/A'
            ),
            'Rank':                '',  # filled below
        })
    tdf = (
        pd.DataFrame(table_rows)
        .assign(Rank=lambda df: [f'#{i+1}' for i in range(len(df))])
    )
    # Reorder columns
    tdf = tdf[['Rank','Airline', f'Predicted ({sym})',
               'vs Your Pick','vs Dataset Avg','Duration','Cost/km']]
    st.dataframe(tdf, use_container_width=True, hide_index=True)

    # ── Summary insight strip ─────────────────────────────────────
    cheapest_price = al_df.iloc[0]['Price']
    priciest_price = al_df.iloc[-1]['Price']
    cheapest_name  = al_df.iloc[0]['Airline']
    priciest_name  = al_df.iloc[-1]['Airline']
    saving_vs_pick = price_d - cheapest_price

    st.markdown(f"""
    <div style="display:flex;gap:14px;margin-top:8px;flex-wrap:wrap;">
        <div style="flex:1;min-width:160px;background:#f0fdf4;border:1px solid #bbf7d0;
                    border-radius:10px;padding:14px 16px;">
            <div style="font-size:0.65rem;font-weight:800;color:#16a34a;
                        text-transform:uppercase;letter-spacing:1px;margin-bottom:4px">
                🏆 Cheapest Option
            </div>
            <div style="font-family:'Syne',sans-serif;font-size:1.1rem;
                        font-weight:900;color:#0f172a">{cheapest_name}</div>
            <div style="font-size:0.85rem;color:#16a34a;font-weight:700">
                {sym}{cheapest_price:,.0f}
            </div>
        </div>
        <div style="flex:1;min-width:160px;background:#eff6ff;border:1px solid #bfdbfe;
                    border-radius:10px;padding:14px 16px;">
            <div style="font-size:0.65rem;font-weight:800;color:#0052cc;
                        text-transform:uppercase;letter-spacing:1px;margin-bottom:4px">
                ⭐ Your Selection
            </div>
            <div style="font-family:'Syne',sans-serif;font-size:1.1rem;
                        font-weight:900;color:#0f172a">{airline}</div>
            <div style="font-size:0.85rem;color:#0052cc;font-weight:700">
                {sym}{price_d:,.0f}
            </div>
        </div>
        <div style="flex:1;min-width:160px;background:#fff7ed;border:1px solid #fed7aa;
                    border-radius:10px;padding:14px 16px;">
            <div style="font-size:0.65rem;font-weight:800;color:#ea580c;
                        text-transform:uppercase;letter-spacing:1px;margin-bottom:4px">
                💡 Potential Saving
            </div>
            <div style="font-family:'Syne',sans-serif;font-size:1.1rem;
                        font-weight:900;color:#0f172a">
                {f"Switch to {cheapest_name}" if saving_vs_pick > 50 else "Best choice!"}
            </div>
            <div style="font-size:0.85rem;color:#ea580c;font-weight:700">
                {f"{sym}{saving_vs_pick:,.0f} cheaper" if saving_vs_pick > 50 else "Already cheapest"}
            </div>
        </div>
        <div style="flex:1;min-width:160px;background:#fdf4ff;border:1px solid #e9d5ff;
                    border-radius:10px;padding:14px 16px;">
            <div style="font-size:0.65rem;font-weight:800;color:#9333ea;
                        text-transform:uppercase;letter-spacing:1px;margin-bottom:4px">
                📊 Price Spread
            </div>
            <div style="font-family:'Syne',sans-serif;font-size:1.1rem;
                        font-weight:900;color:#0f172a">
                {sym}{priciest_price - cheapest_price:,.0f} range
            </div>
            <div style="font-size:0.85rem;color:#9333ea;font-weight:700">
                {cheapest_name} → {priciest_name}
            </div>
        </div>
    </div>
    """, unsafe_allow_html=True)



    # ═══════════════════════════════════════════════════════════════════════════

# Task 3 — SCENARIO COMPARISON
# Why expander: collapses by default so it does not clutter the main result.
# Why batch_predict: all scenario prices computed in ONE model call.
# Why session_state for scenarios: inputs persist across reruns.
# ═══════════════════════════════════════════════════════════════════════════
st.divider()
with st.expander('🔀 Scenario Comparison — Compare up to 3 flight configurations', expanded=False):
    st.caption(
        'Define up to 3 different scenarios (vary airline, stops, month, hour). '
        'Uses the same route as your main search. All prices from the real ML model.'
    )

    # Initialise scenario defaults in session state
    if 'scenarios' not in st.session_state:
        st.session_state.scenarios = [
            {'airline':'IndiGo',    'stops':'non-stop','dep_hour':6,  'month':4,'weekday':0,'label':'Scenario A'},
            {'airline':'SpiceJet',  'stops':'non-stop','dep_hour':10, 'month':4,'weekday':4,'label':'Scenario B'},
            {'airline':'Air India', 'stops':'1 stop',  'dep_hour':18, 'month':5,'weekday':5,'label':'Scenario C'},
        ]

    n_sc = st.radio('Number of scenarios', [2, 3], horizontal=True, index=0, key='_n_sc')
    sc_cols = st.columns(int(n_sc))

    for sci in range(int(n_sc)):
        sc = st.session_state.scenarios[sci]
        with sc_cols[sci]:
            st.markdown(
                f'<div style="font-size:0.65rem;font-weight:800;color:#0052cc;'
                f'text-transform:uppercase;letter-spacing:1.5px;margin-bottom:8px">'
                f'{sc["label"]}</div>', unsafe_allow_html=True
            )
            sc['airline']  = st.selectbox('Airline', AIRLINES,
                                           index=AIRLINES.index(sc['airline']),
                                           key=f'_sc_{sci}_al')
            _vst = VALID_AIRLINE_STOPS.get(sc['airline'], STOPS)
            if sc['stops'] not in _vst: sc['stops'] = _vst[0]
            sc['stops']    = st.selectbox('Stops', _vst,
                                           index=_vst.index(sc['stops']),
                                           key=f'_sc_{sci}_st')
            sc['dep_hour'] = st.slider('Dep Hour', 0, 23, sc['dep_hour'], key=f'_sc_{sci}_hr')
            sc['month']    = st.selectbox('Month', [3,4,5,6],
                                           format_func=lambda m: MONTHS[m],
                                           index=[3,4,5,6].index(sc['month']),
                                           key=f'_sc_{sci}_mo')
            sc['weekday']  = st.selectbox('Weekday', list(range(7)),
                                           format_func=lambda d: WEEKDAYS[d],
                                           index=sc['weekday'], key=f'_sc_{sci}_wd')

    if st.button('⚡ Compare Now', key='_sc_run', type='primary', use_container_width=True):
        _sc_src  = st.session_state.get('source', SOURCES[0])
        _sc_dsts = VALID_DESTINATIONS.get(_sc_src, DESTINATIONS)
        _sc_dst  = st.session_state.get('destination', _sc_dsts[0])

        _active  = st.session_state.scenarios[:int(n_sc)]
        _sc_durs = [predict_duration(_sc_src, _sc_dst, sc['stops']) for sc in _active]
        _sc_combos = [
            dict(airline=sc['airline'], source=_sc_src, destination=_sc_dst,
                 dep_hour=sc['dep_hour'], journey_month=sc['month'],
                 journey_weekday=sc['weekday'], journey_day=15,
                 duration_hours=dur)
            for sc, dur in zip(_active, _sc_durs)
        ]
        _sc_prices = batch_predict_app(_sc_combos, 1)   # batch: 1 call for all scenarios
        _sc_min    = min(_sc_prices)

        # Card results
        res_cols = st.columns(int(n_sc))
        for sci, (sc, p_inr, dur) in enumerate(zip(_active, _sc_prices, _sc_durs)):
            p_d  = round(p_inr * fac, 0)
            best = (p_inr == _sc_min)
            bg   = '#f0fdf4' if best else '#eff6ff'
            bdr  = '#22c55e' if best else '#bfdbfe'
            with res_cols[sci]:
                st.markdown(
                    f'<div style="background:{bg};border:2px solid {bdr};'
                    f'border-radius:12px;padding:16px;text-align:center;">'
                    f'<div style="font-size:0.62rem;font-weight:800;color:#64748b;'
                    f'text-transform:uppercase;letter-spacing:1px">'
                    f'{sc["label"]}{"  🏆" if best else ""}</div>'
                    f'<div style="font-size:0.82rem;color:#0f172a;font-weight:700;margin:4px 0">'
                    f'{sc["airline"]}</div>'
                    f'<div style="font-size:0.7rem;color:#64748b">'
                    f'{sc["stops"]} · {MONTHS[sc["month"]]} · '
                    f'{WEEKDAYS[sc["weekday"]][:3]} · {sc["dep_hour"]:02d}:00 · {dur:.1f}h</div>'
                    f'<div style="font-family:Syne,sans-serif;font-size:2rem;'
                    f'font-weight:900;color:#0052cc;margin:8px 0">{sym}{p_d:,.0f}</div>'
                    f'<div style="font-size:0.68rem;color:#94a3b8">'
                    f'{_sc_src} → {_sc_dst} · 1 pax</div>'
                    f'</div>', unsafe_allow_html=True
                )

        # Comparison bar chart
        _fig_sc = go.Figure(go.Bar(
            x=[sc['label'] for sc in _active],
            y=[round(p*fac) for p in _sc_prices],
            marker_color=['#22c55e' if p==_sc_min else '#0052cc' for p in _sc_prices],
            text=[f'{sym}{round(p*fac):,}' for p in _sc_prices],
            textposition='outside',
            textfont=dict(size=12, family='Syne, sans-serif'),
            width=0.5
        ))
        _fig_sc.update_layout(
            paper_bgcolor='white', plot_bgcolor='white',
            font=dict(family='Plus Jakarta Sans, sans-serif', color='#64748b'),
            yaxis=dict(tickprefix=sym, gridcolor='#e8f0fe'),
            xaxis=dict(tickfont=dict(size=13, family='Syne, sans-serif', color='#0f172a')),
            margin=dict(l=10,r=10,t=30,b=10), height=260, showlegend=False
        )
        st.plotly_chart(_fig_sc, use_container_width=True, config={'displayModeBar': False})

        # Saving tip
        if len(_sc_prices) > 1:
            _sorted = sorted(zip(_sc_prices, _active, _sc_durs))
            _cheap_p, _cheap_sc, _ = _sorted[0]
            _max_p   = _sorted[-1][0]
            _saving  = round((_max_p - _cheap_p)*fac)
            if _saving > 0:
                st.info(
                    f'💡 Switch from **{_sorted[-1][1]["label"]}** to '
                    f'**{_cheap_sc["label"]}** ({_cheap_sc["airline"]}, '
                    f'{_cheap_sc["stops"]}) to save **{sym}{_saving:,}** per passenger.'
                )











Overwriting /content/drive/MyDrive/AirFair-Vista/app/app.py


---
##  Step: Streamlit Server Launch & Process Management

**Why:** `subprocess.Popen` launches Streamlit as a background process while the notebook continues executing — necessary because a blocking `!streamlit run` would freeze the Colab cell. `pkill -f streamlit` first terminates any existing server instances to prevent port conflicts from previous runs. The `--server.headless true` flag suppresses the browser auto-open prompt that would fail in a server environment without a display.

In [ ]:
import subprocess, time, requests

subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(3)

subprocess.Popen(
    ['streamlit', 'run',
     '/content/drive/MyDrive/AirFair-Vista/app/app.py',
     '--server.port',                 '8501',
     '--server.address',              '0.0.0.0',
     '--server.headless',             'true',
     '--server.enableCORS',           'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(6)

try:
    r = requests.get('http://0.0.0.0:8501', timeout=5)
    print(f' Streamlit running! HTTP {r.status_code}')
except Exception as e:
    print(f' {e} — re-run this cell')

 Streamlit running! HTTP 200


---
## 🔷 Step: Public URL via Cloudflare Tunnel

**Why:** Colab notebooks run on Google's servers — not a publicly accessible machine. Cloudflare's `cloudflared` tunnel creates a secure, authenticated reverse proxy from the internet to the local port 8501 where Streamlit is running, generating a temporary public HTTPS URL. This is the standard approach for sharing Streamlit apps from Colab environments without requiring a cloud deployment (Heroku, AWS, etc.), enabling stakeholder demos and faculty review of the live application.

In [ ]:
import subprocess, time, re

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=open('cf.log', 'w'), stderr=subprocess.STDOUT
)
time.sleep(8)
log  = open('cf.log').read()
urls = re.findall(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', log)

if urls:
    print('\n' + '='*55)
    print('   AirFair-Vista is LIVE!')
    print(f'    {urls[0]}')
    print('='*55)
    print('  No password needed. Keep this cell running.')
else:
    time.sleep(5)
    log  = open('cf.log').read()
    urls = re.findall(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', log)
    print(urls[0] if urls else ' Tunnel failed:\n' + log)


   AirFair-Vista is LIVE!
    https://george-remember-putting-danny.trycloudflare.com
  No password needed. Keep this cell running.


---
##  Next Step → Notebook 15: Streamlit Finalization & Enhancements

The core Streamlit application is running. **Notebook 15** adds production-quality enhancements — input validation with user-friendly error messages, price range confidence intervals, airline-price comparison visualisations, a booking calendar heatmap, and mobile-responsive layout — transforming the functional prototype into a polished, deployable product.